# VisionStream AI — Sprint 3: Remote NVIDIA GPU Lab
## 03. ONNX Export & ONNX Runtime CUDA Execution Provider Benchmark

**Tujuan:** Mengekspor canonical baseline YOLOv8n ke format ONNX dan menjalankan benchmark komparatif menggunakan ONNX Runtime dengan backend CUDAExecutionProvider.

> **Catatan Tata Kelola:** Model yang diekspor adalah `yolov8n.pt` kanonikal.

### Step 1: Install Dependencies

In [ ]:
!pip install --quiet ultralytics onnx onnxruntime-gpu

### Step 2: Export YOLOv8n to ONNX Format

In [ ]:
import os
from ultralytics import YOLO

MODEL_PT = "yolov8n.pt"
assert os.path.exists(MODEL_PT), f"{MODEL_PT} must exist in working directory."

model = YOLO(MODEL_PT)
print("Exporting model to ONNX with imgsz=640, dynamic=False, opset=12...")
onnx_path = model.export(format="onnx", imgsz=640, dynamic=False, opset=12, simplify=True)
print(f"Exported ONNX Model path: {onnx_path}")

### Step 3: Run ONNX Runtime GPU Benchmark (CUDAExecutionProvider)

In [ ]:
import time, json
import numpy as np
import onnxruntime as ort

print("Available ONNX Runtime Execution Providers:", ort.get_available_providers())
assert "CUDAExecutionProvider" in ort.get_available_providers(), "CUDAExecutionProvider is NOT available!"

providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
session = ort.InferenceSession("yolov8n.onnx", providers=providers)

input_name = session.get_inputs()[0].name
input_shape = session.get_inputs()[0].shape
print(f"ONNX Model Input Name: {input_name}, Shape: {input_shape}")

# Create normalized float32 tensor (1, 3, 640, 640)
dummy_input = np.random.randn(1, 3, 640, 640).astype(np.float32)

WARMUP_RUNS = 20
MEASURED_RUNS = 200

print(f"Running {WARMUP_RUNS} warm-up iterations on ONNX Runtime CUDA...")
for _ in range(WARMUP_RUNS):
    _ = session.run(None, {input_name: dummy_input})
print("Warm-up complete.")

print(f"Running {MEASURED_RUNS} benchmark iterations on ONNX Runtime CUDA...")
latencies_ms = []
for _ in range(MEASURED_RUNS):
    t0 = time.perf_counter()
    _ = session.run(None, {input_name: dummy_input})
    t1 = time.perf_counter()
    latencies_ms.append((t1 - t0) * 1000.0)

mean_lat = float(np.mean(latencies_ms))
med_lat = float(np.median(latencies_ms))
p95_lat = float(np.percentile(latencies_ms, 95))
fps = float(1000.0 / mean_lat)

print(f"Mean Latency (ONNX CUDA):   {mean_lat:.2f} ms")
print(f"Median Latency (ONNX CUDA): {med_lat:.2f} ms")
print(f"P95 Latency (ONNX CUDA):    {p95_lat:.2f} ms")
print(f"Throughput (ONNX CUDA):     {fps:.2f} FPS")

res_onnx = {
    "provider": "CUDAExecutionProvider",
    "model": "yolov8n.onnx",
    "warmup_runs": WARMUP_RUNS,
    "measured_runs": MEASURED_RUNS,
    "mean_latency_ms": mean_lat,
    "median_latency_ms": med_lat,
    "p95_latency_ms": p95_lat,
    "fps": fps
}

with open("onnx_cuda_benchmark.json", "w") as f:
    json.dump(res_onnx, f, indent=2)
print("Benchmark results exported to onnx_cuda_benchmark.json")